# Some More examples of RE

# Writing a Tokenizer 
(from docs.python.org)

In [ ]:
from typing import NamedTuple
import re

class Token(NamedTuple):
    type: str
    value: int | float | str
    line: int
    column: int

def tokenize(code):
    keywords = {'IF', 'THEN', 'ENDIF', 'FOR', 'NEXT', 'GOSUB', 'RETURN'}
    token_specification = [
        ('NUMBER',   r'\d+(\.\d*)?'),  # Integer or decimal number
        ('ASSIGN',   r':='),           # Assignment operator
        ('END',      r';'),            # Statement terminator
        ('ID',       r'[A-Za-z]+'),    # Identifiers
        ('OP',       r'[+\-*/]'),      # Arithmetic operators
        ('NEWLINE',  r'\n'),           # Line endings
        ('SKIP',     r'[ \t]+'),       # Skip over spaces and tabs
        ('MISMATCH', r'.'),            # Any other character
    ]
    tok_regex = '|'.join('(?P<%s>%s)' % pair for pair in token_specification)
    print(tok_regex)
    line_num = 1
    line_start = 0
    for mo in re.finditer(tok_regex, code):
        kind = mo.lastgroup
        value = mo.group()
        column = mo.start() - line_start
        if kind == 'NUMBER':
            value = float(value) if '.' in value else int(value)
        elif kind == 'ID' and value in keywords:
            kind = value
        elif kind == 'NEWLINE':
            line_start = mo.end()
            line_num += 1
            continue
        elif kind == 'SKIP':
            continue
        elif kind == 'MISMATCH':
            raise RuntimeError(f'{value!r} unexpected on line {line_num}')
        yield Token(kind, value, line_num, column)

statements = '''
    IF quantity THEN
        total := total + price * quantity;
        tax := price * 0.05;
    ENDIF;
'''

for token in tokenize(statements):
    print(token)

#  More Regex Examples

In [ ]:
import re

In [ ]:

xml = """<library>
  <book>
    <title>The Hobbit</title>
    <author>J.R.R. Tolkien</author>
    <year>1937</year>
    <genre>Fantasy</genre>
    <genre>Adventure</genre>
    <genre>Children's</genre>
  </book>
  <book>
    <title>Dune</title>
    <author>Frank Herbert</author>
    <year>1965</year>
    <genre>Science Fiction</genre>
    <genre>Adventure</genre>
  </book>
  <book>
    <title>Beloved</title>
    <author>Toni Morrison</author>
    <year>1987</year>
    <genre>Historical Fiction</genre>
    <genre>Literary Fiction</genre>
  </book>
  <book>
    <title>Frankenstein</title>
    <author>Mary Shelley</author>
    <year>1818</year>
    <genre>Gothic</genre>
    <genre>Horror</genre>
    <genre>Science Fiction</genre>
  </book>
</library>"""

1 Find every title

# Greedy vs non-Greedy

In [ ]:
s = "<b>bold</b>"

print(re.findall(r"<.*>", s))    # ['<b>bold</b>']   greedy: one big match
print(re.findall(r"<.*?>", s) )  # ['<b>', '</b>']   non-greedy: each tag separately

In [ ]:
 s = "<title>Dune</title><title>Emma</title>"
 print(re.findall(r"<title>(.*)</title>", s))
 print(re.findall(r"<title>(.*?)</title>", s))

2. Find a every Genre

4. Title and author pairs. Write one pattern that captures both the title and the author of each book, then print lines like:
The Hobbit by J.R.R. Tolkien
(Hint: use two sets of parentheses. \s* matches any whitespace, including newlines.)

5. Genres by book. Print each book with its genres, like this:
The Hobbit: Fantasy, Adventure, Children's
(Hint: first grab each whole <book>…</book> block using re.DOTALL, then search for the title and genres inside each block.)

6. find by genre. Print the title of every book that has the genre Science Fiction.
Expected output: Dune and Frankenstein

# Nested Tags

In [ ]:


xml = """<library>
  <available>
    <book><title>Dune</title></book>
    <book><title>Beloved</title></book>
  </available>
  <borrowed>
    <book><title>The Hobbit</title></book>
  </borrowed>
</library>"""


7. Find all books that are borrowed

Step 1: get just the <borrowed> section (DOTALL lets . match newlines)
Step 2: search for titles only inside that section

In [ ]:

section = re.search(r"<borrowed>(.*?)</borrowed>", xml, re.DOTALL).group(1)
print(section)
print(re.findall(r"<title>(.*?)</title>", section))   # ['The Hobbit']

For arbitrary nexting Need to go beyond reg expressions to CFG

# Regex Package
support for nexting
import regex as reg

#Parser

import xml.etree.ElementTree as ET

root = ET.fromstring(xml)

print([t.text for t in root.findall("./borrowed/book/title")])   # ['The Hobbit']